In [1]:
import pandas as pd
import numpy as np

In [3]:
# 1. Đọc dữ liệu từ file geography.csv hiện tại
df = pd.read_csv('geography.csv')

# Kiểm tra 5 dòng đầu tiên và thông tin dữ liệu
print("Dữ liệu gốc:")
display(df.head())
print("\nThông tin dữ liệu:")
display(df.info())

Dữ liệu gốc:


,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13
3,15204,Bac Giang,East,District #13
4,15205,Bac Giang,East,District #13



Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39948 entries, 0 to 39947
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   zip       39948 non-null  int64 
 1   city      39948 non-null  object
 2   region    39948 non-null  object
 3   district  39948 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.2+ MB


None

In [ ]:
# 1. Đổi tên các cột cho đúng chuẩn schema yêu cầu
df_geo = df.rename(columns={
    'zip': 'Postal_Code',
    'city': 'City',
    'district': 'State',   # Hoặc dùng 'region' làm State tùy logic nghiệp vụ
    'region': 'Region'
})

# 2. Thêm cột Country (Gán mặc định là 'Vietnam' dựa trên dữ liệu City)
df_geo['Country'] = 'Vietnam'

# 3. Sắp xếp lại dữ liệu theo City và Postal_Code để chuẩn bị đánh index
df_geo = df_geo.sort_values(by=['City', 'Postal_Code']).reset_index(drop=True)

# 4. Loại bỏ các dòng trùng lặp (nếu có) để đảm bảo dữ liệu là duy nhất
df_geo = df_geo.drop_duplicates(subset=['Postal_Code', 'City', 'State', 'Country']) 

# 5. Tạo khóa chính (Primary Key) Geography_ID
# Sinh ID tự tăng từ 1 đến hết
df_geo.insert(0, 'Geography_ID', range(1, len(df_geo) + 1))

# Sắp xếp lại thứ tự cột cho chuyên nghiệp
final_columns = ['Geography_ID', 'City', 'State', 'Region', 'Country', 'Postal_Code']
df_geo = df_geo[final_columns]

print("\nDữ liệu sau khi chuẩn hóa:")
display(df_geo.head())


Dữ liệu sau khi chuẩn hóa:


,Geography_ID,City,State,Region,Country,Postal_Code
0,1,Bac Giang,District #03,East,Vietnam,501
1,2,Bac Giang,District #02,East,Vietnam,1005
2,3,Bac Giang,District #02,East,Vietnam,1014
3,4,Bac Giang,District #02,East,Vietnam,1028
4,5,Bac Giang,District #02,East,Vietnam,1068


In [5]:
# Tổng hợp số lượng Postal Code (zip) theo từng Thành phố (City) và Vùng (Region)
summary_df = df_geo.groupby(['Region', 'City']).agg(
    Total_Postal_Codes=('Postal_Code', 'count')
).reset_index()

# Sắp xếp theo số lượng mã bưu chính giảm dần
summary_df = summary_df.sort_values(by='Total_Postal_Codes', ascending=False)

print("\nBảng tổng hợp số lượng mã bưu chính theo khu vực:")
display(summary_df.head(10))

# Tổng số lượng mã bưu chính duy nhất:
print(f"Tổng số Postal Code: {df_geo['Postal_Code'].nunique()}")


Bảng tổng hợp số lượng mã bưu chính theo khu vực:


,Region,City,Total_Postal_Codes
14,East,Cam Pha,1403
21,East,Phu Ly,1399
23,East,Thai Nguyen,1394
17,East,Hanoi,1376
19,East,Nam Dinh,1370
15,East,Ha Long,1357
12,East,Bac Giang,1347
16,East,Hai Phong,1346
13,East,Bac Ninh,1346
22,East,Son Tay,1344


Tổng số Postal Code: 39948


In [10]:
# Lưu đè lại vào file geography.csv (hoặc đặt tên mới để không làm mất file gốc)
output_file = 'geography_silver.csv' # Khuyến nghị lưu ra file mới 

df_geo.to_csv(output_file, index=False, encoding='utf-8')

print(f"👉🏿 Đã xử lý và lưu thành công {len(df_geo)} bản ghi vào file {output_file}!")

👉🏿 Đã xử lý và lưu thành công 39948 bản ghi vào file geography_silver.csv!
